*Made by Phuc*

## MỤC TIÊU ##

1. Đọc file `data/processed/final_sales.csv`
2. Tính Iceberg Cube bằng thuật toán BUC top-down
3. Phân cụm khách hàng bằng RFM và K-Means
4. Xuất file cho app và báo cáo

In [52]:
# Import thư viện cần dùng
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Cấu hình đường dẫn và tham số

Input sau bước tiền xử lý:

`data/processed/final_sales.csv`

Output của phần này là:

- `outputs/iceberg_cube.csv`
- `outputs/customer_clusters.csv`
- `outputs/model_metrics.json`

In [53]:
from pathlib import Path

# Xác định project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "final_sales.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ICEBERG_OUTPUT_PATH = OUTPUT_DIR / "iceberg_cube.csv"
CLUSTER_OUTPUT_PATH = OUTPUT_DIR / "customer_clusters.csv"
METRICS_OUTPUT_PATH = OUTPUT_DIR / "model_metrics.json"

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Output dir:", OUTPUT_DIR)
print("Input exists:", INPUT_PATH.exists())

Project root: /Users/dzafuc/Documents/Data Mining/final-data-mining-project
Input path: /Users/dzafuc/Documents/Data Mining/final-data-mining-project/data/processed/final_sales.csv
Output dir: /Users/dzafuc/Documents/Data Mining/final-data-mining-project/outputs
Input exists: True


## 2. Đọc dữ liệu

File input cần có các cột tối thiểu:

- `order_id`
- `customer_unique_id`
- `order_purchase_timestamp`
- `order_month`
- `customer_state`
- `product_category_name`
- `payment_type`
- `total_amount`
- `review_score`
- `delivery_days`

In [54]:
def load_sale_view(path: Path) -> pd.DataFrame:
    """Đọc dữ liệu SALE view và kiểm tra các cột cần có."""
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file {path}. Vui lòng kiểm tra lại đường dẫn."
        )

    df = pd.read_csv(path)

    required_columns = [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_month",
        "customer_state",
        "product_category_name",
        "payment_type",
        "total_amount",
        "review_score",
        "delivery_days",
    ]

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"File input đang thiếu các cột: {missing_columns}")

    return df


df = load_sale_view(INPUT_PATH)
df.head()

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_month,customer_state,seller_state,product_category_name,payment_type,price,freight_value,total_amount,review_score,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10,SP,SP,utilidades_domesticas,credit_card,29.99,8.72,38.71,4.0,6.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10,SP,SP,utilidades_domesticas,voucher,29.99,8.72,38.71,4.0,6.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10,SP,SP,utilidades_domesticas,voucher,29.99,8.72,38.71,4.0,6.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,2018-07,BA,SP,perfumaria,boleto,118.70,22.76,141.46,4.0,12.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,2018-08,GO,SP,automotivo,credit_card,159.90,19.22,179.12,5.0,9.0


In [55]:
# Kiểm tra nhanh dữ liệu
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118310 entries, 0 to 118309
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  118310 non-null  object 
 1   customer_id               118310 non-null  object 
 2   customer_unique_id        118310 non-null  object 
 3   order_purchase_timestamp  118310 non-null  object 
 4   order_month               118310 non-null  object 
 5   customer_state            118310 non-null  object 
 6   seller_state              118310 non-null  object 
 7   product_category_name     116601 non-null  object 
 8   payment_type              118307 non-null  object 
 9   price                     118310 non-null  float64
 10  freight_value             118310 non-null  float64
 11  total_amount              118310 non-null  float64
 12  review_score              117332 non-null  float64
 13  delivery_days             115721 non-null  f

## 3. Chuẩn bị dữ liệu cho BUC

BUC dùng `support_count` để pruning. Vì `count` có tính giảm dần khi đi sâu xuống các dimension con, nếu một nhóm không đạt `min_sup` thì các nhóm con của nó cũng không cần xét tiếp.

In [56]:
def prepare_cube_data(df: pd.DataFrame, dimensions: list[str]) -> pd.DataFrame:
    """Làm sạch đơn giản các cột dùng cho BUC."""
    cube_df = df.copy()

    for dim in dimensions:
        cube_df[dim] = cube_df[dim].fillna("unknown").astype(str)

    numeric_columns = ["total_amount", "review_score", "delivery_days"]
    for col in numeric_columns:
        cube_df[col] = pd.to_numeric(cube_df[col], errors="coerce")

    cube_df["total_amount"] = cube_df["total_amount"].fillna(0)
    cube_df["review_score"] = cube_df["review_score"].fillna(cube_df["review_score"].median())

    # delivery_days âm là dữ liệu không hợp lý, chuyển thành missing rồi điền median
    cube_df.loc[cube_df["delivery_days"] < 0, "delivery_days"] = np.nan
    cube_df["delivery_days"] = cube_df["delivery_days"].fillna(cube_df["delivery_days"].median())

    return cube_df


cube_df = prepare_cube_data(df, BUC_DIMENSIONS)
cube_df[BUC_DIMENSIONS + ["total_amount", "review_score", "delivery_days"]].head()

,order_month,customer_state,product_category_name,payment_type,total_amount,review_score,delivery_days
0,2017-10,SP,utilidades_domesticas,credit_card,38.71,4.0,6.0
1,2017-10,SP,utilidades_domesticas,voucher,38.71,4.0,6.0
2,2017-10,SP,utilidades_domesticas,voucher,38.71,4.0,6.0
3,2018-07,BA,perfumaria,boleto,141.46,4.0,12.0
4,2018-08,GO,automotivo,credit_card,179.12,5.0,9.0


## 4. Thuật toán BUC Top-Down Iceberg Cube

Cách hoạt động:

1. Bắt đầu từ cell tổng quát `ALL`.
2. Chia dữ liệu theo từng dimension.
3. Nếu một partition có `support_count < min_sup`, dừng nhánh đó.
4. Nếu partition đạt `min_sup`, lưu cell và tiếp tục đi sâu xuống dimension sau.
5. Kết quả cuối cùng chỉ gồm các cube cell đạt điều kiện iceberg.

In [57]:
from typing import Dict, List, Optional

def aggregate_cube_cell(data: pd.DataFrame, prefix: dict, dimensions: list[str]) -> dict:
    """Tạo một cube cell từ partition hiện tại."""
    record = {}

    for dim in dimensions:
        record[dim] = prefix.get(dim, "ALL")

    record["cuboid_level"] = sum(1 for dim in dimensions if record[dim] != "ALL")
    record["support_count"] = int(len(data))
    record["order_count"] = int(data["order_id"].nunique())
    record["total_sales"] = float(data["total_amount"].sum())
    record["avg_review_score"] = float(data["review_score"].mean())
    record["avg_delivery_days"] = float(data["delivery_days"].mean())

    return record


def buc_top_down(
    data: pd.DataFrame,
    dimensions: List[str],
    min_sup: int,
    start_dim: int = 0,
    prefix: Optional[Dict[str, object]] = None,
    results: Optional[List[dict]] = None,
) -> List[dict]:
    """
    Tính Iceberg Cube bằng cơ chế BUC top-down.

    data: partition hiện tại
    dimensions: danh sách dimension
    min_sup: ngưỡng support tối thiểu
    start_dim: dimension bắt đầu xét
    prefix: các giá trị dimension đã chọn
    results: danh sách cube cell đạt điều kiện iceberg
    """
    if prefix is None:
        prefix = {}
    if results is None:
        results = []

    # Iceberg pruning
    if len(data) < min_sup:
        return results

    # Lưu cell hiện tại
    results.append(aggregate_cube_cell(data, prefix, dimensions))

    # Đi xuống các dimension con
    for dim_index in range(start_dim, len(dimensions)):
        dim = dimensions[dim_index]

        for value, partition in data.groupby(dim, dropna=False):
            if len(partition) >= min_sup:
                prefix[dim] = value
                buc_top_down(
                    data=partition,
                    dimensions=dimensions,
                    min_sup=min_sup,
                    start_dim=dim_index + 1,
                    prefix=prefix,
                    results=results,
                )
                prefix.pop(dim, None)

    return results


buc_results = buc_top_down(
    data=cube_df,
    dimensions=BUC_DIMENSIONS,
    min_sup=MIN_SUP,
)

iceberg_cube = pd.DataFrame(buc_results)

iceberg_cube = iceberg_cube.sort_values(
    by=["cuboid_level", "support_count", "total_sales"],
    ascending=[True, False, False],
).reset_index(drop=True)

iceberg_cube.to_csv(ICEBERG_OUTPUT_PATH, index=False)

print(f"Đã xuất file: {ICEBERG_OUTPUT_PATH}")
print(f"Số cube cell đạt điều kiện iceberg: {len(iceberg_cube)}")
iceberg_cube.head(20)

Đã xuất file: /Users/dzafuc/Documents/Data Mining/final-data-mining-project/outputs/iceberg_cube.csv
Số cube cell đạt điều kiện iceberg: 3538


,order_month,customer_state,product_category_name,payment_type,cuboid_level,support_count,order_count,total_sales,avg_review_score,avg_delivery_days
0,ALL,ALL,ALL,ALL,0,118310,98666,16643731.30,4.039397,8.714318
1,ALL,ALL,ALL,credit_card,1,87258,75991,12776455.36,4.041051,8.736437
2,ALL,SP,ALL,ALL,1,49865,41375,6234533.82,4.132779,5.120766
3,ALL,ALL,ALL,boleto,1,23018,19614,2859446.84,4.032453,8.692806
4,ALL,RJ,ALL,ALL,1,15425,12762,2247128.32,3.828979,11.253225
5,ALL,MG,ALL,ALL,1,13718,11544,1928571.09,4.093016,8.238081
6,ALL,ALL,cama_mesa_banho,ALL,1,11988,9417,1327662.02,3.903654,9.149233
7,ALL,ALL,beleza_saude,ALL,1,10032,8836,1491397.76,4.144338,8.439195
8,2017-11,ALL,ALL,ALL,1,9096,7451,1232074.30,3.853012,10.248901
9,ALL,ALL,esporte_lazer,ALL,1,9004,7720,1205197.85,4.113616,8.829742


In [58]:
# Xem các nhóm chi tiết có doanh thu cao nhất
iceberg_cube.sort_values("total_sales", ascending=False).head(20)

,order_month,customer_state,product_category_name,payment_type,cuboid_level,support_count,order_count,total_sales,avg_review_score,avg_delivery_days
0,ALL,ALL,ALL,ALL,0,118310,98666,16643731.30,4.039397,8.714318
1,ALL,ALL,ALL,credit_card,1,87258,75991,12776455.36,4.041051,8.736437
2,ALL,SP,ALL,ALL,1,49865,41375,6234533.82,4.132779,5.120766
113,ALL,SP,ALL,credit_card,2,36688,31833,4751155.64,4.137947,5.096789
3,ALL,ALL,ALL,boleto,1,23018,19614,2859446.84,4.032453,8.692806
4,ALL,RJ,ALL,ALL,1,15425,12762,2247128.32,3.828979,11.253225
5,ALL,MG,ALL,ALL,1,13718,11544,1928571.09,4.093016,8.238081
114,ALL,RJ,ALL,credit_card,2,11693,10188,1763271.51,3.832378,11.180279
115,ALL,MG,ALL,credit_card,2,10224,8965,1497028.73,4.100548,8.261150
7,ALL,ALL,beleza_saude,ALL,1,10032,8836,1491397.76,4.144338,8.439195


## 5. Tạo dữ liệu RFM cho phân cụm

RFM gồm:

- `Recency`: khách hàng mua gần đây hay đã lâu.
- `Frequency`: số lần mua hàng.
- `Monetary`: tổng số tiền khách hàng đã chi.

Có thêm một số biến hỗ trợ như `avg_order_value`, `avg_review_score`, `avg_delivery_days`.

In [59]:
def build_customer_rfm(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo bảng RFM ở cấp khách hàng."""
    rfm_df = df.copy()

    rfm_df["order_purchase_timestamp"] = pd.to_datetime(
        rfm_df["order_purchase_timestamp"], errors="coerce"
    )

    rfm_df = rfm_df.dropna(subset=["order_purchase_timestamp", "customer_unique_id"])

    numeric_columns = ["total_amount", "review_score", "delivery_days"]
    for col in numeric_columns:
        rfm_df[col] = pd.to_numeric(rfm_df[col], errors="coerce")

    
    rfm_df.loc[rfm_df["delivery_days"] < 0, "delivery_days"] = np.nan
    rfm_df["total_amount"] = rfm_df["total_amount"].fillna(0)
    rfm_df["review_score"] = rfm_df["review_score"].fillna(rfm_df["review_score"].median())
    rfm_df["delivery_days"] = rfm_df["delivery_days"].fillna(rfm_df["delivery_days"].median())

    snapshot_date = rfm_df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

    customer_rfm = (
        rfm_df.groupby("customer_unique_id")
        .agg(
            recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
            frequency=("order_id", "nunique"),
            monetary=("total_amount", "sum"),
            avg_order_value=("total_amount", "mean"),
            avg_review_score=("review_score", "mean"),
            avg_delivery_days=("delivery_days", "mean"),
        )
        .reset_index()
    )

    return customer_rfm


customer_rfm = build_customer_rfm(df)
customer_rfm.head()

,customer_unique_id,recency,frequency,monetary,avg_order_value,avg_review_score,avg_delivery_days
0,0000366f3b9a7992bf8c76cfdf3221e2,116,1,141.90,141.90,5.0,4.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,119,1,27.19,27.19,4.0,1.0
2,0000f46a3911fa3c0805444483337064,542,1,86.22,86.22,3.0,23.0
3,0000f6ccb0745a6a4b88665a16c9f078,326,1,43.62,43.62,4.0,19.0
4,0004aac84e0df4da2b147fca70cf8255,293,1,196.89,196.89,5.0,11.0


In [60]:
customer_rfm[CLUSTER_FEATURES].describe()

,recency,frequency,monetary,avg_order_value,avg_review_score,avg_delivery_days
count,95420.000000,95420.000000,95420.000000,95420.000000,95420.000000,95420.000000
mean,243.600377,1.034018,174.426025,146.748097,4.109025,8.850089
std,153.160320,0.211234,264.680442,199.021092,1.323843,8.656553
min,1.000000,1.000000,10.070000,9.341429,1.000000,0.000000
25%,119.000000,1.000000,64.010000,57.800000,4.000000,4.000000
50%,224.000000,1.000000,110.915000,96.690000,5.000000,7.000000
75%,353.000000,1.000000,188.940000,163.240000,5.000000,11.000000
max,729.000000,16.000000,13664.080000,6929.310000,5.000000,205.000000


## 6. Chọn số cụm bằng Silhouette Score

Vì K-Means dùng khoảng cách, dữ liệu cần được chuẩn hóa trước khi phân cụm.

In [61]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import time

SILHOUETTE_SAMPLE_SIZE = 10000

silhouette_scores = {}
inertias = {}

n_rows = X_scaled.shape[0]
sample_size = min(SILHOUETTE_SAMPLE_SIZE, n_rows)

print(f"Số dòng dùng cho clustering: {n_rows}")
print(f"Số dòng dùng để tính silhouette: {sample_size}")

for k in K_RANGE:
    start_time = time.time()
    print(f"Đang chạy K-Means với k={k}...")

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    inertias[k] = float(model.inertia_)

    silhouette_scores[k] = float(
        silhouette_score(
            X_scaled,
            labels,
            sample_size=sample_size,
            random_state=RANDOM_STATE
        )
    )

    elapsed = time.time() - start_time
    print(f"k={k} | inertia={inertias[k]:.2f} | silhouette={silhouette_scores[k]:.4f} | time={elapsed:.2f}s")

best_k = max(silhouette_scores, key=silhouette_scores.get)

print("\nSilhouette scores:")
for k, score in silhouette_scores.items():
    print(f"k={k}: {score:.4f}")

print(f"\nSố cụm được chọn: k={best_k}")

Số dòng dùng cho clustering: 95420
Số dòng dùng để tính silhouette: 10000
Đang chạy K-Means với k=2...
k=2 | inertia=456320.21 | silhouette=0.2253 | time=1.33s
Đang chạy K-Means với k=3...
k=3 | inertia=368526.10 | silhouette=0.2462 | time=1.52s
Đang chạy K-Means với k=4...
k=4 | inertia=302272.61 | silhouette=0.2673 | time=1.49s
Đang chạy K-Means với k=5...
k=5 | inertia=262477.06 | silhouette=0.2443 | time=1.37s
Đang chạy K-Means với k=6...
k=6 | inertia=233574.35 | silhouette=0.2571 | time=1.35s
Đang chạy K-Means với k=7...
k=7 | inertia=206947.63 | silhouette=0.2522 | time=1.14s

Silhouette scores:
k=2: 0.2253
k=3: 0.2462
k=4: 0.2673
k=5: 0.2443
k=6: 0.2571
k=7: 0.2522

Số cụm được chọn: k=4


## 7. Huấn luyện K-Means và xuất kết quả

In [62]:
final_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
customer_rfm["cluster"] = final_kmeans.fit_predict(X_scaled)

# PCA dùng để vẽ 2D trên app hoặc trong báo cáo
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_result = pca.fit_transform(X_scaled)
customer_rfm["pca_1"] = pca_result[:, 0]
customer_rfm["pca_2"] = pca_result[:, 1]

cluster_profile = (
    customer_rfm.groupby("cluster")[CLUSTER_FEATURES]
    .mean()
    .round(2)
    .reset_index()
)

cluster_counts = (
    customer_rfm["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="customer_count")
)

cluster_profile = cluster_profile.merge(cluster_counts, on="cluster", how="left")
cluster_profile

,cluster,recency,frequency,monetary,avg_order_value,avg_review_score,avg_delivery_days,customer_count
0,0,246.32,1.00,69.89,63.88,4.60,6.83,44634
1,1,251.57,1.00,159.73,129.83,1.52,16.35,14629
2,2,226.20,2.11,348.72,126.01,4.16,8.49,2913
3,3,237.96,1.00,305.97,267.27,4.58,8.29,33244


## 8. Đặt tên cụm

Tên cụm có thể chỉnh thủ công sau khi xem `cluster_profile`. Notebook sẽ tự tạo tên gợi ý để app dễ hiển thị.

In [65]:
def assign_cluster_names(profile: pd.DataFrame) -> dict:
    """
    Đặt tên cụm dựa trên đặc điểm trung bình của từng cụm.

    Ý nghĩa chính:
    - Khách hàng giá trị cao: monetary và frequency cao, recency thấp
    - Khách hàng giá trị thấp: monetary và frequency thấp
    - Khách hàng trải nghiệm thấp: review thấp, delivery_days cao
    - Khách hàng tiềm năng: các cụm còn lại
    """
    profile = profile.copy()

    # Điểm giá trị khách hàng
    # monetary cao tốt, frequency cao tốt, recency thấp tốt
    profile["value_score"] = (
        profile["monetary"].rank(ascending=True)
        + profile["frequency"].rank(ascending=True)
        + profile["recency"].rank(ascending=False)
    )

    # Điểm trải nghiệm xấu
    # review thấp là xấu, delivery_days cao là xấu
    profile["bad_experience_score"] = (
        profile["avg_review_score"].rank(ascending=False)
        + profile["avg_delivery_days"].rank(ascending=True)
    )

    # Mặc định các cụm là khách hàng tiềm năng
    cluster_names = {
        int(row["cluster"]): "Khách hàng tiềm năng"
        for _, row in profile.iterrows()
    }

    # Cụm có trải nghiệm thấp nhất
    bad_experience_cluster = int(
        profile.sort_values("bad_experience_score", ascending=True)
        .iloc[0]["cluster"]
    )

    # Cụm có giá trị cao nhất
    high_value_cluster = int(
        profile.sort_values("value_score", ascending=False)
        .iloc[0]["cluster"]
    )

    # Cụm có giá trị thấp nhất
    low_value_cluster = int(
        profile.sort_values("value_score", ascending=True)
        .iloc[0]["cluster"]
    )

    # Gán tên cụm
    cluster_names[bad_experience_cluster] = "Khách hàng trải nghiệm thấp"
    cluster_names[high_value_cluster] = "Khách hàng có giá trị cao"

    # Tránh ghi đè nếu cụm giá trị thấp trùng với cụm trải nghiệm thấp
    if low_value_cluster not in [bad_experience_cluster, high_value_cluster]:
        cluster_names[low_value_cluster] = "Khách hàng có giá trị thấp"

    # Nếu vẫn còn cụm chưa có tên đặc biệt thì giữ là khách hàng tiềm năng
    return cluster_names


cluster_name_map = assign_cluster_names(cluster_profile)

customer_rfm["cluster_name"] = customer_rfm["cluster"].map(cluster_name_map)
cluster_profile["cluster_name"] = cluster_profile["cluster"].map(cluster_name_map)

customer_rfm.to_csv(CLUSTER_OUTPUT_PATH, index=False)

print(f"Đã xuất file: {CLUSTER_OUTPUT_PATH}")
cluster_profile

Đã xuất file: /Users/dzafuc/Documents/Data Mining/final-data-mining-project/outputs/customer_clusters.csv


,cluster,recency,frequency,monetary,avg_order_value,avg_review_score,avg_delivery_days,customer_count,cluster_name
0,0,246.32,1.00,69.89,63.88,4.60,6.83,44634,Khách hàng trải nghiệm thấp
1,1,251.57,1.00,159.73,129.83,1.52,16.35,14629,Khách hàng tiềm năng
2,2,226.20,2.11,348.72,126.01,4.16,8.49,2913,Khách hàng có giá trị cao
3,3,237.96,1.00,305.97,267.27,4.58,8.29,33244,Khách hàng tiềm năng


## 9. Lưu file metrics

File `model_metrics.json` dùng cho app, README hoặc phần kết quả trong báo cáo.

In [66]:
metrics = {
    "iceberg_cube": {
        "algorithm": "BUC top-down",
        "dimensions": BUC_DIMENSIONS,
        "min_sup": MIN_SUP,
        "input_rows": int(len(cube_df)),
        "output_cells": int(len(iceberg_cube)),
        "output_file": str(ICEBERG_OUTPUT_PATH),
    },
    "clustering": {
        "algorithm": "K-Means",
        "features": CLUSTER_FEATURES,
        "k_range": list(K_RANGE),
        "best_k": int(best_k),
        "silhouette_scores": {str(k): v for k, v in silhouette_scores.items()},
        "inertias": {str(k): v for k, v in inertias.items()},
        "pca_explained_variance_ratio": [float(v) for v in pca.explained_variance_ratio_],
        "cluster_names": {str(k): v for k, v in cluster_name_map.items()},
        "output_file": str(CLUSTER_OUTPUT_PATH),
    },
    "cluster_profile": cluster_profile.to_dict(orient="records"),
}

with open(METRICS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

print(f"Đã xuất file: {METRICS_OUTPUT_PATH}")
metrics

Đã xuất file: /Users/dzafuc/Documents/Data Mining/final-data-mining-project/outputs/model_metrics.json


{'iceberg_cube': {'algorithm': 'BUC top-down',
  'dimensions': ['order_month',
   'customer_state',
   'product_category_name',
   'payment_type'],
  'min_sup': 50,
  'input_rows': 118310,
  'output_cells': 3538,
  'output_file': '/Users/dzafuc/Documents/Data Mining/final-data-mining-project/outputs/iceberg_cube.csv'},
 'clustering': {'algorithm': 'K-Means',
  'features': ['recency',
   'frequency',
   'monetary',
   'avg_order_value',
   'avg_review_score',
   'avg_delivery_days'],
  'k_range': [2, 3, 4, 5, 6, 7],
  'best_k': 4,
  'silhouette_scores': {'2': 0.2252901141929506,
   '3': 0.24623633742156908,
   '4': 0.2673437807655161,
   '5': 0.24431990239549964,
   '6': 0.25713734673675714,
   '7': 0.25218372986541765},
  'inertias': {'2': 456320.21246219723,
   '3': 368526.09787728894,
   '4': 302272.61228848435,
   '5': 262477.0636948544,
   '6': 233574.3543423584,
   '7': 206947.63370688635},
  'pca_explained_variance_ratio': [0.32624305181755947, 0.21069796562904283],
  'cluster_na